# Script for comparing different CNN models 

In [19]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,mean_absolute_error,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

mne.set_log_level("CRITICAL")

# Loading in Data

In [20]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_2904.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [ ]:
np.shape(features_all_store)
 

# Functions

In [21]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [22]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(2, activation='linear'))


    return model  # Return the compiled model

In [23]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


 
    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        num_classes,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [24]:
# single head CNN model for number of contractions  
def CNN_model_contraction_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output")(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [25]:
# single head CNN model for number of contractions  
def CNN_model_duration_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(1, activation='relu', name="zygo_output")(x)

    # output head for Corr
    out_corr = Dense(1, activation='relu', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [26]:
def CNN_model_fourhead_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # --- Shared CNN backbone ---
    x = Conv1D(32, kernel_size=3, activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    x = Flatten()(x)

    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # zygo branch 
    zygo_branch = Dense(32, activation='relu')(shared)
    zygo_branch = Dropout(0.3)(zygo_branch)

    zygo_count_output = Dense(
        num_classes,
        activation='softmax',
        name='zygo_count_output'
    )(zygo_branch)

    zygo_duration_output = Dense(
        1,
        activation='linear',
        name='zygo_duration_output'
    )(zygo_branch)

    # corr branch 
    corr_branch = Dense(32, activation='relu')(shared)
    corr_branch = Dropout(0.2)(corr_branch)

    corr_count_output = Dense(
        num_classes,
        activation='softmax',
        name='corr_count_output'
    )(corr_branch)

    corr_duration_output = Dense(
        1,
        activation='linear',
        name='corr_duration_output'
    )(corr_branch)

    # --- Model ---
    model = Model(
        inputs=inputs,
        outputs=[
            zygo_count_output,
            zygo_duration_output,
            corr_count_output,
            corr_duration_output
        ]
    )

    return model

# Single Channel Training 

## Single Head Count 

In [27]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [28]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_test,y_test)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 

# redefine fresh model 
#final_model = CNN_model_contraction(input_shape, num_classes, feature_num)
#final_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10


2026-05-11 15:34:38.191140: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 14s 45ms/step - loss: 5.5626 - accuracy: 0.7308 - val_loss: 0.8876 - val_accuracy: 0.8556
Epoch 2/10
286/286 [==============================] - 11s 39ms/step - loss: 2.4773 - accuracy: 0.7859 - val_loss: 1.0972 - val_accuracy: 0.8643
Epoch 3/10
286/286 [==============================] - 10s 35ms/step - loss: 3.4079 - accuracy: 0.8030 - val_loss: 1.6635 - val_accuracy: 0.8652
Epoch 4/10
286/286 [==============================] - 13s 47ms/step - loss: 1.7661 - accuracy: 0.8550 - val_loss: 2.1320 - val_accuracy: 0.8635
Epoch 5/10
286/286 [==============================] - 14s 49ms/step - loss: 0.9430 - accuracy: 0.8923 - val_loss: 2.2490 - val_accuracy: 0.8639
Epoch 6/10
286/286 [==============================] - 11s 37ms/step - loss: 0.9368 - accuracy: 0.9064 - val_loss: 4.0705 - val_accuracy: 0.8617
Epoch 7/10
286/286 [==============================] - 10s 36ms/step - loss: 1.3872 - accuracy: 0.9038 - val_loss: 4.2106 - val_accuracy: 0.8477
Epo

2026-05-11 15:36:39.647943: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 13s 43ms/step - loss: 2.4234 - accuracy: 0.7995 - val_loss: 0.7398 - val_accuracy: 0.8508
Epoch 2/10
286/286 [==============================] - 11s 39ms/step - loss: 1.0836 - accuracy: 0.8323 - val_loss: 1.3548 - val_accuracy: 0.8604
Epoch 3/10
286/286 [==============================] - 11s 37ms/step - loss: 0.9944 - accuracy: 0.8587 - val_loss: 1.3525 - val_accuracy: 0.8543
Epoch 4/10
286/286 [==============================] - 15s 53ms/step - loss: 0.7938 - accuracy: 0.8915 - val_loss: 2.1268 - val_accuracy: 0.8565
Epoch 5/10
286/286 [==============================] - 14s 47ms/step - loss: 0.4819 - accuracy: 0.9236 - val_loss: 3.5267 - val_accuracy: 0.8416
Epoch 6/10
286/286 [==============================] - 13s 45ms/step - loss: 0.5211 - accuracy: 0.9340 - val_loss: 3.0363 - val_accuracy: 0.8403
Epoch 7/10
286/286 [==============================] - 13s 46ms/step - loss: 0.5578 - accuracy: 0.9455 - val_loss: 5.6275 - val_accuracy: 0.8420
Epo

...

286/286 [==============================] - 13s 43ms/step - loss: 5.6032 - accuracy: 0.7340 - val_loss: 0.9820 - val_accuracy: 0.8735
Epoch 2/10
286/286 [==============================] - 11s 38ms/step - loss: 3.1655 - accuracy: 0.7852 - val_loss: 1.4189 - val_accuracy: 0.8635
Epoch 3/10
286/286 [==============================] - 10s 34ms/step - loss: 2.9788 - accuracy: 0.8106 - val_loss: 1.9159 - val_accuracy: 0.8827
Epoch 4/10
286/286 [==============================] - 12s 41ms/step - loss: 2.1752 - accuracy: 0.8436 - val_loss: 2.3086 - val_accuracy: 0.8761
Epoch 5/10
286/286 [==============================] - 11s 40ms/step - loss: 1.6468 - accuracy: 0.8684 - val_loss: 2.5462 - val_accuracy: 0.8783
Epoch 6/10
286/286 [==============================] - 10s 34ms/step - loss: 1.3884 - accuracy: 0.8853 - val_loss: 1.8902 - val_accuracy: 0.8757
Epoch 7/10
286/286 [==============================] - 11s 37ms/step - loss: 2.3211 - accuracy: 0.8732 - val_loss: 4.3787 - val_accuracy: 0.8639
Epo

2026-05-11 15:40:37.151473: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 13s 42ms/step - loss: 2.7449 - accuracy: 0.7994 - val_loss: 0.6856 - val_accuracy: 0.8354
Epoch 2/10
286/286 [==============================] - 12s 41ms/step - loss: 1.1086 - accuracy: 0.8330 - val_loss: 1.5714 - val_accuracy: 0.8477
Epoch 3/10
286/286 [==============================] - 9s 33ms/step - loss: 1.2613 - accuracy: 0.8578 - val_loss: 1.5561 - val_accuracy: 0.8433
Epoch 4/10
286/286 [==============================] - 11s 38ms/step - loss: 1.1225 - accuracy: 0.8862 - val_loss: 2.4287 - val_accuracy: 0.8416
Epoch 5/10
286/286 [==============================] - 9s 31ms/step - loss: 0.7336 - accuracy: 0.9194 - val_loss: 2.9571 - val_accuracy: 0.8368
Epoch 6/10
286/286 [==============================] - 9s 31ms/step - loss: 0.6210 - accuracy: 0.9342 - val_loss: 4.1778 - val_accuracy: 0.8359
Epoch 7/10
286/286 [==============================] - 10s 35ms/step - loss: 0.7545 - accuracy: 0.9331 - val_loss: 4.6636 - val_accuracy: 0.8407
Epoch 

2026-05-11 15:42:26.805821: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 12s 41ms/step - loss: 14.9314 - accuracy: 0.6629 - val_loss: 2.4249 - val_accuracy: 0.8634
Epoch 2/10
286/286 [==============================] - 15s 52ms/step - loss: 6.0911 - accuracy: 0.7656 - val_loss: 3.0506 - val_accuracy: 0.8586
Epoch 3/10
286/286 [==============================] - 14s 51ms/step - loss: 5.5176 - accuracy: 0.7905 - val_loss: 3.9203 - val_accuracy: 0.8678
Epoch 4/10
286/286 [==============================] - 14s 48ms/step - loss: 3.6858 - accuracy: 0.8299 - val_loss: 4.5287 - val_accuracy: 0.8651
Epoch 5/10
286/286 [==============================] - 12s 41ms/step - loss: 3.0099 - accuracy: 0.8443 - val_loss: 4.9179 - val_accuracy: 0.8590
Epoch 6/10
286/286 [==============================] - 14s 49ms/step - loss: 3.1023 - accuracy: 0.8554 - val_loss: 7.2497 - val_accuracy: 0.8538
Epoch 7/10
286/286 [==============================] - 13s 45ms/step - loss: 2.5852 - accuracy: 0.8788 - val_loss: 8.2776 - val_accuracy: 0.8616
Ep

In [34]:
# cross validation results 
avgScores = np.mean(cvScores_contraction)
stdScores = np.std(cvScores_contraction)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

Average KFold Cross Validation Score: 93.35434079170227
Standard Deviation KFold Cross Validation Score: 0.8958111405953845
90/90 [==============================] - 1s 9ms/step
Model scores---------------
Training Accuracy : 0.9483543417366946
Test Accuracy : 0.8995098039215687
Training F1 Score : 0.9456460791060658
Test F1 Score : 0.8860688685247529


# Count by Muscle Groups 

In [33]:
zygo_ind_train = np.where(idx_train < 7140)
corr_ind_train = np.where(idx_train >= 7140)  
zygo_ind_test = np.where(idx_test < 7140)
corr_ind_test = np.where(idx_test >= 7140)  

# remake splits 
X_train_full_corr = X[corr_ind_train]
X_train_full_zygo = X[zygo_ind_train]

X_test_corr = X[corr_ind_test]
X_test_zygo = X[zygo_ind_test]

y_train_full_corr = y[corr_ind_train]
y_train_full_zygo = y[zygo_ind_train]

y_test_corr = y[corr_ind_test]
y_test_zygo = y[zygo_ind_test]


# full training results (test data not seen during cross val)
y_pred_train_corr = model_contraction.predict(X_train_full_corr)  
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)   

y_pred_train_zygo = model_contraction.predict(X_train_full_zygo)  
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)   

# Predict on test data
y_pred_test_zygo = model_contraction.predict(X_test_zygo)   
y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   

y_pred_test_corr= model_contraction.predict(X_test_corr)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   

# Calculate accuracy
accuracy_training_corr = accuracy_score(y_train_full_corr, y_pred_train_corr)   
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)  

accuracy_training_zygo = accuracy_score(y_train_full_zygo, y_pred_train_zygo)   
accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)  

# Calculate F1 score
f1_training_corr = f1_score(y_train_full_corr, y_pred_train_corr, average='weighted')  
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')  

f1_training_zygo = f1_score(y_train_full_zygo, y_pred_train_zygo, average='weighted')  
f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')  


# Print accuracy and F1 score
print("Corru Scores------------------------------")  
print("Training Accuracy :", accuracy_training_corr) 
print("Test Accuracy :", accuracy_test_corr)  
print("Training F1 Score :", f1_training_corr)  
print("Test F1 Score :", f1_test_corr) 

print("Zygo Scores------------------------------")  
print("Training Accuracy :", accuracy_training_zygo)  
print("Test Accuracy :", accuracy_test_zygo)  
print("Training F1 Score :", f1_training_zygo)   
print("Test F1 Score :", f1_test_zygo) 

45/45 [==============================] - 1s 11ms/step
Corru Scores------------------------------
Training Accuracy : 0.9438596491228071
Test Accuracy : 0.9263888888888889
Training F1 Score : 0.9408817379821
Test F1 Score : 0.9218694576633338
Zygo Scores------------------------------
Training Accuracy : 0.9404262753319357
Test Accuracy : 0.9364406779661016
Training F1 Score : 0.9374054002759934
Test F1 Score : 0.930966150832331


In [37]:
confusion_matrix(y_corr, y_pred_test_corr)
confusion_matrix(y_zygo, y_pred_test_zygo)

ValueError: Found input variables with inconsistent numbers of samples: [7140, 1440]

# Single Head Duration

In [ ]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 10 
k = 1
 
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]


    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mae', metrics=['mae'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_test,y_test)
    cvScores_duration.append(scores[1])

    k += 1 
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

In [ ]:
np.shape(y_train_full) 
np.shape(y) 

In [ ]:
# cross validation results 
cvScores_duration = np.array(cvScores_duration)
cvScores_duration /= 100 

avgScores = np.mean(cvScores_duration)
stdScores = np.std(cvScores_duration)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)
 

Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mae'],metrics=['accuracy', 'mae'] ) #think about metric 
    model_history_kfold = model_twohead.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                            validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                            epochs=epoch_num)
    
    scores = model_twohead.evaluate(X_test,[y_test[:,0], y_test[:,1]])
    metrics = dict(zip(model_twohead.metrics_names, scores))

    cvScores_dur.append(metrics['duration_output_mae'])
    cvScores_contr.append(metrics['count_output_accuracy'] * 100)


    cvScores.append(scores)
 
    

  
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

In [ ]:
cvScores

In [ ]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores,axis=0)[3]
stdScores_dur = np.std(cvScores,axis=0)[3]

avgScores_contr= np.mean(cvScores,axis=0)[6]
stdScores_contr = np.std(cvScores,axis=0)[6]
 

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

y_pred_train_dur = model_twohead.predict(X_train_full) 
y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


## Two Channel 

Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model_contraction_multichan(input_shape, num_classes,feature_num)
    model.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'sparse_categorical_crossentropy',
        'corr_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'zygo_output': ['accuracy'],
        'corr_output': ['accuracy']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]])) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores.append(scores[1] * 100)

    k += 1 
    

model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) 

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

Single Head Duration 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_dur = CNN_model_duration_multichan(input_shape, num_classes,feature_num)
    model_dur.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'mae',
        'corr_output': 'mae'
    },
    metrics={
        'zygo_output': ['mae'],
        'corr_output': ['mae']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_dur.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]])) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_dur.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores.append(scores)

    k += 1 
    

model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) 

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores,axis=0)[1]
stdScores = np.std(cvScores,axis=0)[1]

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_dur.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_dur.predict(X_test)


baseline_zygo = np.mean(y_train_full[:,0])
mae_baseline_zygo = np.mean(np.abs(y_train_full[:,0] - baseline_zygo))

baseline_corr = np.mean(y_train_full[:,1])
mae_baseline_corr = np.mean(np.abs(y_train_full[:,1] - baseline_corr))
  
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,0], y_pred_train_zygo)   
mae_test_dur_zygo = mean_absolute_error(y_test[:,0], y_pred_test_zygo)  

mae_training_dur_corr = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo)   
mae_test_dur_corr= mean_absolute_error(y_test[:,1], y_pred_test_zygo)  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)
print("----------------------") 
print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)

Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y_contr = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


y_dur = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])


y = np.column_stack((y_contr, y_dur))
y = y[:, [0, 2, 1, 3]]

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
np.shape(y)

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_fourhead=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
    model_fourhead.compile(
    optimizer='adam',
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mse'
    },
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_fourhead.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=epoch_num, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)

    
    scores = model_fourhead.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores_fourhead.append(scores)

    k += 1 
    


In [ ]:

# Cross-validation results
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")


y_pred_train = model_fourhead.predict(X_train_full)
y_pred_test = model_fourhead.predict(X_test)

y_pred_train_zygo_count, y_pred_train_zygo_dur, y_pred_train_corr_count, y_pred_train_corr_dur = y_pred_train
y_pred_test_zygo_count, y_pred_test_zygo_dur, y_pred_test_corr_count, y_pred_test_corr_dur = y_pred_test


# baselines
baseline_zygo = np.mean(y_train_full[:,1])
baseline_corr = np.mean(y_train_full[:,3])

mae_baseline_zygo = np.mean(np.abs(y_train_full[:,1] - baseline_zygo))
mae_baseline_corr = np.mean(np.abs(y_train_full[:,3] - baseline_corr))

# MAE
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo_dur)
mae_test_dur_zygo = mean_absolute_error(y_test[:,1], y_pred_test_zygo_dur)

mae_training_dur_corr = mean_absolute_error(y_train_full[:,3], y_pred_train_corr_dur)
mae_test_dur_corr = mean_absolute_error(y_test[:,3], y_pred_test_corr_dur)

print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)

print("----------------------") 

print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)



y_pred_train_zygo_count = np.argmax(y_pred_train_zygo_count, axis=1)
y_pred_train_corr_count = np.argmax(y_pred_train_corr_count, axis=1)

y_pred_test_zygo_count = np.argmax(y_pred_test_zygo_count, axis=1)
y_pred_test_corr_count = np.argmax(y_pred_test_corr_count, axis=1)

# true labels
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]

y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# accuracy
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo_count)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr_count)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo_count)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr_count)

# f1
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo_count, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr_count, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo_count, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr_count, average='weighted')



print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n-------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)


print("\n-------- Average (Counts) --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)

print("\n-------- Average (Duration MAE) --------")
print("Training MAE:", (mae_training_dur_zygo + mae_training_dur_corr) / 2)
print("Test MAE:", (mae_test_dur_zygo + mae_test_dur_corr) / 2)

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
